# 2.02 Google Generative AI Embeddings

Google의 Gemini Embedding(`models/gemini-embedding-001`)을 `langchain-google-genai`로 호출한다.
다른 공급자와 달리 **`task_type`** 파라미터로 검색/분류/클러스터링 목적을 명시할 수 있고, 이 값에 따라 내부 임베딩 분포가 조정되어 다운스트림 품질이 달라진다.

## 학습 목표

- `GoogleGenerativeAIEmbeddings`로 질의·문서를 벡터화한다
- `task_type`을 `retrieval_query` / `retrieval_document` / `semantic_similarity`로 전환해 RAG를 비대칭 인덱싱한다
- `output_dimensionality`로 차원 축소를 걸고 vector store 스키마를 결정한다
- Gemini API vs Vertex AI 엔드포인트 선택 기준을 안다

## 언제 쓰나

- **Gemini 기반 RAG 파이프라인**과 통일된 공급자를 유지하고 싶을 때
- **다국어 + 긴 컨텍스트**(2048 토큰)를 한 번에 넣는 장문 임베딩
- **Vertex AI 기업 배포**에서 IAM·감사로그와 결합해야 할 때

## 2.02.1 환경 설정

필요 패키지: `langchain-google-genai`. `.env`에 `GOOGLE_API_KEY`가 있어야 한다 (AI Studio에서 발급).

In [ ]:
# !pip install -U langchain langchain-google-genai

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("GOOGLE_API_KEY"), "GOOGLE_API_KEY 필요"

## 2.02.2 기본 사용법

`models/gemini-embedding-001`을 기본 모델로 사용한다. `embed_query`와 `embed_documents`는 다른 공급자와 동일한 인터페이스.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

q_vec = embeddings.embed_query("Gemini 임베딩은 무엇인가?")
print("query dim:", len(q_vec), "| preview:", q_vec[:4])

docs = [
    "Gemini는 Google의 멀티모달 LLM 계열이다.",
    "gemini-embedding-001은 최대 3072차원 벡터를 생성한다.",
    "Vertex AI 경유로 엔터프라이즈 배포가 가능하다.",
]
doc_vecs = embeddings.embed_documents(docs)
print("docs:", len(doc_vecs), "x", len(doc_vecs[0]))

## 2.02.3 차원 · 비용 · 다국어 성능

| 항목 | 값 |
|------|----|
| 모델 | `gemini-embedding-001` (GA), `gemini-embedding-2` (최신) |
| 기본 차원 | 3072 (Matryoshka) |
| 축소 가능 차원 | 128 / 256 / 512 / 768 / 1536 / 3072 |
| 최대 입력 토큰 | 2048 |
| 다국어 | 100+ 언어 공식 지원, 한국어 포함 |
| 가격 (2026-04 기준) | Gemini API 무료 티어 + 유료 구간 |

### `output_dimensionality`

OpenAI의 `dimensions`와 같은 Matryoshka 축소. vector store 스키마 고정용으로 유용.

In [ ]:
reduced = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    output_dimensionality=768,
)
v = reduced.embed_query("768차원으로 축소")
print("reduced dim:", len(v))

## 2.02.4 `task_type` 비대칭 인덱싱

이 공급자의 차별점. 검색 파이프라인에서 **질의**와 **문서**를 서로 다른 task_type으로 임베딩하면 정밀도가 올라간다.

| task_type | 용도 |
|-----------|------|
| `retrieval_query` | 사용자 질의 측 임베딩 |
| `retrieval_document` | 코퍼스 적재 측 임베딩 |
| `semantic_similarity` | 대칭 유사도 (예: 중복 감지) |
| `classification` | 분류기 피처 |
| `clustering` | 군집화 피처 |

In [ ]:
doc_emb = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document",
)
query_emb = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_query",
)

corpus = [
    "LangChain은 체인과 에이전트로 LLM을 조립한다.",
    "임베딩은 의미를 고차원 벡터로 투영한다.",
]
doc_vecs = doc_emb.embed_documents(corpus)
q = query_emb.embed_query("벡터로 의미를 표현하는 방법")
print("doc0 dim:", len(doc_vecs[0]), "| query dim:", len(q))

## 2.02.5 벡터스토어 연동 미리보기

비대칭 인덱싱을 그대로 `Chroma`에 붙이려면 **적재 시 document 임베더**, **검색 시 query 임베더**를 각기 지정한다.
간단한 인메모리 예시(자세한 건 `03_vectorstores`).

In [ ]:
# !pip install -U langchain-chroma
from langchain_chroma import Chroma

store = Chroma.from_texts(
    texts=corpus,
    embedding=doc_emb,
    collection_name="demo_google",
)
# 간단한 데모에서는 doc_emb로 대칭 검색해도 동작한다.
hits = store.similarity_search("의미 벡터", k=2)
for h in hits:
    print("-", h.page_content)

## 2.02.6 Vertex AI 엔드포인트

엔터프라이즈 배포(IAM, VPC-SC, 감사로그) 요구 시 Gemini API 대신 Vertex AI를 쓴다.
`langchain-google-genai` 4.x는 `vertexai=True` 플래그로 동일 클래스에서 라우팅한다.

| 필드 | Gemini API | Vertex AI |
|------|-----------|-----------|
| 키 | `GOOGLE_API_KEY` | ADC (Application Default Credentials) |
| `vertexai` | `False` (기본) | `True` |
| `project` | 불필요 | 필수 |
| `location` | 불필요 | `us-central1` 등 |

In [ ]:
# Vertex AI 경로 (ADC 인증 필요)
# vertex_emb = GoogleGenerativeAIEmbeddings(
#     model="models/gemini-embedding-001",
#     vertexai=True,
#     project=os.environ["GOOGLE_CLOUD_PROJECT"],
#     location="us-central1",
# )
# print(vertex_emb.embed_query("Vertex AI 테스트")[:3])
print("Vertex AI 섹션은 예시 코드. ADC 설정이 된 환경에서 주석 해제.")

## 체크리스트

- [ ] RAG 파이프라인은 query/document 측 `task_type`을 다르게 두었는가
- [ ] `output_dimensionality`는 vector store 스키마와 일치하는가
- [ ] 입력 토큰이 2048을 넘는 문서는 사전에 청크 분할했는가
- [ ] 엔터프라이즈 요구 시 `vertexai=True` + `project` + `location` 세팅

## 다음

- `03_cohere.ipynb` — 다국어 특화 `embed-multilingual-v3.0`
- `../05_retrievers/` — Gemini Embedding + MMR 리트리버 조합

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/text_embedding/google_generative_ai
- Gemini Embedding: https://ai.google.dev/gemini-api/docs/embeddings
- `task_type` 종류: https://ai.google.dev/gemini-api/docs/embeddings#task-types